In [1]:
import pandas as pd

In [2]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier



In [3]:
df = pd.read_csv("../data/raw/accidents.csv")

In [4]:
df.head()

,grav,sexe,trajet,secu1,senc,obsm,choc,manv,motor,lum,...,nuit_sans_eclairage,surface_dangereuse,vitesse_x_collision,age_x_securite,agglo_x_vitesse,saison,est_mois_vacances,deux_roues,poids_lourd,vehicule_leger
0,3,1.0,1.0,1.0,1.0,2.0,1.0,1.0,5.0,2.0,...,0,0,80.0,0.0,80.0,3.0,0,0,0,0
1,1,1.0,1.0,1.0,1.0,9.0,3.0,17.0,1.0,2.0,...,0,0,80.0,43.0,80.0,3.0,0,0,0,1
2,4,1.0,5.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,...,0,0,0.0,38.0,80.0,3.0,0,0,0,1
3,3,1.0,5.0,1.0,2.0,2.0,1.0,9.0,1.0,1.0,...,0,0,0.0,28.0,80.0,3.0,0,0,0,1
4,1,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,...,0,0,0.0,26.0,100.0,2.0,1,0,0,1


In [9]:
X = df.drop(columns = "grav")
y = df["grav"]
y = y - 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

In [10]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(405173, 33) (101294, 33) (405173,) (101294,)


In [11]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

# 1. On pointe vers notre serveur local
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# 2. On crée (ou on réouvre) une expérience nommée
mlflow.set_experiment("accidents-gravite")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1771847293555, experiment_id='1', last_update_time=1771847293555, lifecycle_stage='active', name='accidents-gravite', tags={}, workspace='default'>

In [16]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

# On ouvre un "run" = une session d'enregistrement
with mlflow.start_run(run_name="XGBoost-baseline"):
    
    # 1. Entraînement (ton code existant)
    model = XGBClassifier(max_depth=3, n_estimators=100, learning_rate=0.1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    # 2. Logger les hyperparamètres
    mlflow.log_params(model.get_params())
    
    # 3. Logger les métriques
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average="weighted"))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,  average='weighted', multi_class='ovr'))
    mlflow.log_metric("precision", precision_score(y_test, y_pred, average="weighted"))
    mlflow.log_metric("recall", recall_score(y_test, y_pred, average="weighted"))
    
    # 4. Logger le modèle sérialisé
    mlflow.xgboost.log_model(model, artifact_path="model")

2026/02/23 16:00:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/5987ce7c31c445e690f040511d46d83c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:

models = {
    "LogisticRegression": (LogisticRegression(max_iter=1000), mlflow.sklearn),
    "XGBoost":            (XGBClassifier(n_estimators=100), mlflow.xgboost),
    "LightGBM":           (LGBMClassifier(n_estimators=100), mlflow.lightgbm),
    "RandomForest":       (RandomForestClassifier(n_estimators=20), mlflow.sklearn),
}

for run_name, (model, mlflow_module) in models.items():
    with mlflow.start_run(run_name=run_name):
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        mlflow.log_params(model.get_params())
        
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("f1", f1_score(y_test, y_pred, average="weighted"))
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,  average='weighted', multi_class='ovr'))
        mlflow.log_metric("precision", precision_score(y_test, y_pred, average="weighted"))
        mlflow.log_metric("recall", recall_score(y_test, y_pred, average="weighted"))
        
        mlflow_module.log_model(model, artifact_path="model")
        
        print(f"✅ {run_name} loggé")

c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/02/23 16:43:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 16:43:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe al

✅ LogisticRegression loggé
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/3c8885e109d045d9bb0e15682b98ee17
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/02/23 16:44:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ XGBoost loggé
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/239fd83b890a4ef190cf66d63710acb7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040384 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 462
[LightGBM] [Info] Number of data points in the train set: 405173, number of used features: 33
[LightGBM] [Info] Start training from score -0.856198
[LightGBM] [Info] Start training from score -3.619780
[LightGBM] [Info] Start training from score -1.886693
[LightGBM] [Info] Start training from score -0.924159


2026/02/23 16:44:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 16:44:31 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ LightGBM loggé
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/1/runs/36b3d5c2244349fcba9eff66ed1d7d09
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/02/23 16:45:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 16:45:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ RandomForest loggé
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/1/runs/b766d220146a4ddaa89808e402da6ac1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [20]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("accidents-gravite-tuning")

2026/02/24 10:55:24 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1771926924047, experiment_id='2', last_update_time=1771926924047, lifecycle_stage='active', name='accidents-gravite-tuning', tags={}, workspace='default'>

In [21]:
configs = [
    {
        "max_depth": 3,
        "n_estimators": 100,
        "learning_rate": 0.1,
        "subsample": 1.0
    },
    {
        "max_depth": 5,
        "n_estimators": 200,
        "learning_rate": 0.05,
        "subsample": 0.8
    },
    {
        "max_depth": 7,
        "n_estimators": 300,
        "learning_rate": 0.01,
        "subsample": 0.8
    },
]

for i, params in enumerate(configs):
    with mlflow.start_run(run_name=f"XGBoost-config-{i+1}"):
        
        model = XGBClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        # Logger les paramètres
        mlflow.log_params(params)
        
        # Logger les métriques
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
        mlflow.log_metric("precision", precision_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("recall", recall_score(y_test, y_pred, average='weighted'))
        
        print(f"✅ Config {i+1} loggée")

✅ Config 1 loggée
🏃 View run XGBoost-config-1 at: http://127.0.0.1:5000/#/experiments/2/runs/2072512c26334f7ab00d1ac19f98aaef
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
✅ Config 2 loggée
🏃 View run XGBoost-config-2 at: http://127.0.0.1:5000/#/experiments/2/runs/4cdb909229744fbdb2915cd76affba6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
✅ Config 3 loggée
🏃 View run XGBoost-config-3 at: http://127.0.0.1:5000/#/experiments/2/runs/6434656b2c774eeb8d0dd8a8e763e187
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [22]:
# 📝 DOCUMENTATION — Meilleure configuration identifiée
# =====================================================
# Expérience  : accidents-gravite-tuning
# Modèle      : XGBoost
# 
# Config retenue :
#   - learning_rate : 0.05
#   - max_depth     : 5
#   - n_estimators  : 200
#   - subsample     : 0.8
#
# Métriques obtenues :
#   - Accuracy  : 0.652
#   - F1        : 0.631
#   - ROC AUC   : 0.827
#
# Justification :
#   Meilleur F1 et ROC AUC parmi les 3 configs testées.
#   Config 3 (max_depth=7) plus complexe mais moins performante
#   → signe d'overfitting. Config 2 retenue comme meilleur compromis.
#
# Note : le baseline (params défaut) reste légèrement meilleur en accuracy
#        (0.662 vs 0.652). Le tuning GridSearch/Optuna (Jour 3) permettra
#        d'explorer plus de combinaisons pour dépasser ce baseline.
# =====================================================

In [23]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # important : évite les problèmes d'affichage en dehors de jupyter

from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix
from sklearn.preprocessing import label_binarize
import os

# Dossier temporaire pour sauvegarder les fichiers avant de les logger
os.makedirs("tmp_artifacts", exist_ok=True)

with mlflow.start_run(run_name="XGBoost-config2-avec-artefacts"):
    
    # 1. Entraînement avec la meilleure config
    params = {
        "max_depth": 5,
        "n_estimators": 200,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "random_state": 42
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # 2. Paramètres et métriques
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))

    # 3. Matrice de confusion
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    ax.set_title("Matrice de confusion — XGBoost Config 2")
    fig.savefig("tmp_artifacts/confusion_matrix.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/confusion_matrix.png")

    # 4. Courbes ROC (une par classe)
    classes = sorted(y_test.unique())
    y_test_bin = label_binarize(y_test, classes=classes)

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, classe in enumerate(classes):
        RocCurveDisplay.from_predictions(
            y_test_bin[:, i],
            y_proba[:, i],
            name=f"Classe {classe}",
            ax=ax
        )
    ax.set_title("Courbes ROC par classe — XGBoost Config 2")
    fig.savefig("tmp_artifacts/roc_curves.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/roc_curves.png")

    # 5. Feature names (métadonnées utiles)
    feature_names = X_train.columns.tolist()
    with open("tmp_artifacts/feature_names.txt", "w") as f:
        f.write("\n".join(feature_names))
    mlflow.log_artifact("tmp_artifacts/feature_names.txt")

    # 6. Modèle
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("✅ Run avec artefacts loggé !")

2026/02/24 11:39:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Run avec artefacts loggé !
🏃 View run XGBoost-config2-avec-artefacts at: http://127.0.0.1:5000/#/experiments/2/runs/f0c29a3a0fe74b90bcb74db0046b4be3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [24]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # important : évite les problèmes d'affichage en dehors de jupyter

from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix
from sklearn.preprocessing import label_binarize
import os

# Dossier temporaire pour sauvegarder les fichiers avant de les logger
os.makedirs("tmp_artifacts", exist_ok=True)

with mlflow.start_run(run_name="XGBoost-config2-avec-artefacts"):
    
    # 1. Entraînement avec la meilleure config
    params = {
        "max_depth": 3,
        "n_estimators": 100,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "random_state": 42
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # 2. Paramètres et métriques
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))

    # 3. Matrice de confusion
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    ax.set_title("Matrice de confusion — XGBoost Config 2")
    fig.savefig("tmp_artifacts/confusion_matrix.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/confusion_matrix.png")

    # 4. Courbes ROC (une par classe)
    classes = sorted(y_test.unique())
    y_test_bin = label_binarize(y_test, classes=classes)

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, classe in enumerate(classes):
        RocCurveDisplay.from_predictions(
            y_test_bin[:, i],
            y_proba[:, i],
            name=f"Classe {classe}",
            ax=ax
        )
    ax.set_title("Courbes ROC par classe — XGBoost Config 2")
    fig.savefig("tmp_artifacts/roc_curves.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/roc_curves.png")

    # 5. Feature names (métadonnées utiles)
    feature_names = X_train.columns.tolist()
    with open("tmp_artifacts/feature_names.txt", "w") as f:
        f.write("\n".join(feature_names))
    mlflow.log_artifact("tmp_artifacts/feature_names.txt")

    # 6. Modèle
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("✅ Run avec artefacts loggé !")

2026/02/24 11:44:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Run avec artefacts loggé !
🏃 View run XGBoost-config2-avec-artefacts at: http://127.0.0.1:5000/#/experiments/2/runs/c787cef00cc4429aabe287b3931b1d0f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [28]:
from mlflow import MlflowClient

client = MlflowClient()

# Cherche dans TOUTES tes expériences le meilleur recall
best_run = mlflow.search_runs(
     experiment_names=[
        "accidents-gravite",
        "accidents-gravite-tuning"
    ],
    order_by=["metrics.recall DESC"]
).iloc[0]

print(f"✅ Meilleur run    : {best_run['tags.mlflow.runName']}")
print(f"   Recall         : {best_run['metrics.recall']}")
print(f"   ROC AUC        : {best_run['metrics.roc_auc']}")
print(f"   Run ID         : {best_run['run_id']}")

✅ Meilleur run    : XGBoost
   Recall         : 0.6621023950085888
   ROC AUC        : 0.8376957510322283
   Run ID         : 239fd83b890a4ef190cf66d63710acb7


In [29]:
# 1. Construire l'URI du modèle à partir du run_id
model_uri = f"runs:/{best_run['run_id']}/model"

# 2. Enregistrer dans le Registry
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="xgboost-gravite-accidents"
)

print(f"✅ Modèle enregistré !")
print(f"   Nom     : {registered_model.name}")
print(f"   Version : {registered_model.version}")

Successfully registered model 'xgboost-gravite-accidents'.
2026/02/24 13:42:22 WARNING mlflow.tracking._model_registry.fluent: Run with id 239fd83b890a4ef190cf66d63710acb7 has no artifacts at artifact path 'model', registering model based on models:/m-604f4eff2cc9490eae37de47349700be instead
2026/02/24 13:42:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost-gravite-accidents, version 1


✅ Modèle enregistré !
   Nom     : xgboost-gravite-accidents
   Version : 1


Created version '1' of model 'xgboost-gravite-accidents'.


In [30]:
client = MlflowClient()

# Description du modèle
client.update_registered_model(
    name="xgboost-gravite-accidents",
    description="Modèle XGBoost de prédiction de gravité d'accidents routiers. "
                "Sélectionné sur le recall pour minimiser les faux négatifs sur cas graves."
)

# Description de cette version
client.update_model_version(
    name="xgboost-gravite-accidents",
    version=registered_model.version,
    description=f"Baseline XGBoost — recall={best_run['metrics.recall']:.3f} | "
                f"roc_auc={best_run['metrics.roc_auc']:.3f}"
)

print("✅ Description ajoutée !")

✅ Description ajoutée !


In [31]:
client.transition_model_version_stage(
    name="xgboost-gravite-accidents",
    version=registered_model.version,
    stage="Staging"
)

print(f"✅ Modèle v{registered_model.version} passé en Staging !")

✅ Modèle v1 passé en Staging !


C:\Users\ins expertise\AppData\Local\Temp\ipykernel_3984\763122162.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [32]:
# Charger depuis le Registry
model_loaded = mlflow.pyfunc.load_model(
    model_uri=f"models:/xgboost-gravite-accidents/Staging"
)

# Faire une prédiction de test
y_pred_test = model_loaded.predict(X_test[:5])
print(f"✅ Modèle rechargé depuis le Registry !")
print(f"   Prédictions test : {y_pred_test}")

✅ Modèle rechargé depuis le Registry !
   Prédictions test : [3 0 0 3 3]


In [33]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from mlflow import MlflowClient

mlflow.set_experiment("accidents-gravite-gridsearch")

# 1. Définir la grille
param_grid = {
    "max_depth":     [3, 5, 7],
    "n_estimators":  [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
}

# 2. Lancer GridSearchCV
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=42, subsample=0.8),
    param_grid=param_grid,
    scoring="recall_weighted",   # notre métrique métier
    cv=cv,
    n_jobs=-1,                   # utilise tous les cœurs disponibles
    verbose=1
)

grid_search.fit(X_train, y_train)

# 3. Logger chaque combinaison comme un run MLflow distinct
for i, params in enumerate(grid_search.cv_results_["params"]):
    with mlflow.start_run(run_name=f"GridSearch-{i+1}"):

        # Réentraîner sur tout X_train avec ces params
        model = XGBClassifier(**params, subsample=0.8, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        # Logger
        mlflow.log_params(params)
        mlflow.log_metric("recall", recall_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,
                                                     multi_class='ovr', average='weighted'))
        # Score CV (ce que GridSearch a vraiment optimisé)
        mlflow.log_metric("cv_recall", grid_search.cv_results_["mean_test_score"][i])

print(f"✅ {len(grid_search.cv_results_['params'])} runs loggés !")
print(f"   Meilleurs params : {grid_search.best_params_}")
print(f"   Meilleur CV recall : {grid_search.best_score_:.4f}")

2026/02/24 15:42:13 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite-gridsearch' does not exist. Creating a new experiment.


Fitting 3 folds for each of 27 candidates, totalling 81 fits
🏃 View run GridSearch-1 at: http://127.0.0.1:5000/#/experiments/3/runs/10f7b2f0a3884af696ff39dc004a1485
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-2 at: http://127.0.0.1:5000/#/experiments/3/runs/eeb9f4ae72f04f4987330ff8fb886922
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-3 at: http://127.0.0.1:5000/#/experiments/3/runs/f287a24f9027457186acfad517058eed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-4 at: http://127.0.0.1:5000/#/experiments/3/runs/21e7f9eb8b784669ab8e2149ce56c909
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-5 at: http://127.0.0.1:5000/#/experiments/3/runs/f242c8bfa39f45bdad8a4a8c5f9d55a8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-6 at: http://127.0.0.1:5000/#/experiments/3/runs/deb7cb6184af49798ef40658c7800384
🧪 View experime